# Executive summary

This document computes a conservative valuation for the collection represented by
`test_results/comic_valuations_20251020_141403.csv`. It uses the "Low Estimate ($)"
column as a conservative baseline and identifies items that look suspicious or
potentially misidentified (these require manual review).

Key outputs provided:
- Conservative total (sum of low estimates)
- Conservative total excluding suspicious/misidentified items
- List of flagged items with reasons (low identification confidence, low valuation confidence, suspicious series/title text such as "Collected" / "Volume", or explicit Batman-related suspects)
- Top items by best estimate (for reference)

Please review flagged items carefully (especially the Batman example you mentioned).
If an item is misidentified (e.g., a collected edition labeled as an original single-issue),
update that row (series/title/issue) and rerun this report.

---

## Load data and compute summary


In [ ]:
#| label: load-and-summarize
#| echo: true
import pandas as pd
from pathlib import Path
import textwrap

csv_path = Path("test_results/comic_valuations_20251020_141403.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found at: {csv_path.resolve()}")

# Read CSV
df = pd.read_csv(csv_path, dtype=str)

# Normalize/clean numeric columns
num_cols_map = {
    'Low Estimate ($)': 'low',
    'Best Estimate ($)': 'best',
    'High Estimate ($)': 'high',
    'Valuation Confidence': 'val_conf',
    'Identification Confidence': 'id_conf'
}

for col, short in num_cols_map.items():
    if col in df.columns:
        df[short] = pd.to_numeric(df[col].str.replace(',', '').fillna('0'), errors='coerce').fillna(0.0)
    else:
        df[short] = 0.0

# Basic counts and sums
total_items = len(df)
sum_low = df['low'].sum()
sum_best = df['best'].sum()
sum_high = df['high'].sum()

print(f"Total rows read: {total_items}")
print(f"Conservative total (sum of low estimates): ${sum_low:,.2f}")
print(f"Aggregate best estimate (sum of best estimates): ${sum_best:,.2f}")
print(f"Aggregate high estimate (sum of high estimates): ${sum_high:,.2f}")

## Flagging suspicious / misidentified items

We flag items with any of the following heuristics:

- Identification confidence is low (default threshold: < 0.60)
- Valuation confidence is low (default threshold: < 0.40)
- Series or Title contains suspicious terms: "collected", "collected adventures", "volume", "vol." (these often indicate trade paperback/collected editions, not single-issue)
- Series or Title mentions "Batman" together with "collected" or "volume" (user-reported misidentification)
- Extremely unusual valuations relative to median (optional)

These are heuristics: flagged items should be reviewed manually.


In [ ]:
#| label: flag-items
import re

# Thresholds (conservative defaults)
ID_CONF_THRESHOLD = 0.60
VAL_CONF_THRESHOLD = 0.40

def find_suspicious_text(s):
    if pd.isna(s):
        return False
    s = s.lower()
    return any(term in s for term in ['collected', 'collected adventures', 'volume ', 'vol. ', 'tpb', 'trade paperback'])

# Initialize flags column
df['flags'] = ''

# Low identification confidence
mask_low_id = df['id_conf'] < ID_CONF_THRESHOLD
df.loc[mask_low_id, 'flags'] += 'low_ident_conf;'

# Low valuation confidence
mask_low_val = df['val_conf'] < VAL_CONF_THRESHOLD
df.loc[mask_low_val, 'flags'] += 'low_val_conf;'

# Suspicious textual markers (collected editions, volumes, trades)
series_col = df.get('Series', pd.Series(['']*len(df)))
title_col = df.get('Title', pd.Series(['']*len(df)))

mask_susp_text = series_col.fillna('').str.contains(r'collected|volume|vol\\.|tpb|trade paperback', case=False, regex=True) | \
                  title_col.fillna('').str.contains(r'collected|volume|vol\\.|tpb|trade paperback', case=False, regex=True)
df.loc[mask_susp_text, 'flags'] += 'suspicious_text;'

# Batman-specific heuristic: "Batman" combined with "Collected" or "Volume"
mask_batman_collected = (series_col.fillna('').str.contains('batman', case=False, na=False) | title_col.fillna('').str.contains('batman', case=False, na=False)) & \
                        (series_col.fillna('').str.contains(r'collected|volume|vol\\.|tpb', case=False, na=False) | title_col.fillna('').str.contains(r'collected|volume|vol\\.|tpb', case=False, na=False))
df.loc[mask_batman_collected, 'flags'] += 'batman_collected_mismatch;'

# Very large outlier best vs low (best much higher than low) - optional
# We'll flag if best is 10x low AND best >= $100
mask_outlier = (df['low'] > 0) & (df['best'] / df['low'] >= 10) & (df['best'] >= 100)
df.loc[mask_outlier, 'flags'] += 'large_discrepancy;'

# Create a boolean masked column for flagged rows
df['is_flagged'] = df['flags'].str.strip().astype(bool)

flagged_count = df['is_flagged'].sum()
print()
print(f"Flagged items count: {flagged_count}")

---

## Conservative totals excluding flagged items


In [ ]:
#| label: conservative-excluding
conservative_total_all = df['low'].sum()
conservative_total_unflagged = df.loc[~df['is_flagged'], 'low'].sum()
conservative_total_high_conf = df.loc[(df['id_conf'] >= ID_CONF_THRESHOLD) & (df['val_conf'] >= VAL_CONF_THRESHOLD), 'low'].sum()

print("Conservative totals (sum of low estimates):")
print(f"- All items: ${conservative_total_all:,.2f}")
print(f"- Excluding flagged items: ${conservative_total_unflagged:,.2f}")
print(f"- Only high-confidence items (id_conf >= {ID_CONF_THRESHOLD}, val_conf >= {VAL_CONF_THRESHOLD}): ${conservative_total_high_conf:,.2f}")

pct_flagged = (flagged_count / total_items * 100) if total_items else 0.0
print()
print(f"Flagged items represent {pct_flagged:.1f}% of the dataset.")

## Top items by best estimate (for manual review)


In [ ]:
#| label: top-by-best
top_n = 12
top_items = df.sort_values('best', ascending=False).head(top_n)
top_table = top_items[['Image Filename', 'Series', 'Title', 'Issue Number', 'low', 'best', 'high', 'val_conf', 'id_conf', 'flags']].copy()
top_table = top_table.rename(columns={'low': 'Low ($)', 'best': 'Best ($)', 'high': 'High ($)', 'val_conf':'ValConf', 'id_conf':'IdConf'})
print("Top items by Best Estimate (top 12):")
print(top_table.to_markdown(index=False))

---

## Detailed flagged items (sample)


In [ ]:
#| label: flagged-sample
if flagged_count:
    sample_flagged = df[df['is_flagged']].copy()
    # show the key columns and reason
    cols_display = ['Image Filename', 'Series', 'Title', 'Issue Number', 'Low Estimate ($)', 'Best Estimate ($)', 'Valuation Confidence', 'Identification Confidence', 'flags']
    # ensure columns exist
    cols_display = [c for c in cols_display if c in df.columns]
    print(f"Showing up to 50 flagged items (total flagged: {flagged_count}):")
    display_df = sample_flagged[cols_display].head(50)
    # Use simple textual display
    print(display_df.to_markdown(index=False))
else:
    print("No flagged items found.")

## Special note: user-reported Batman misidentification

You reported that an item "Batman — The Collected Adventures Volume 1" was incorrectly valued as if it were the first Batman single-issue from the 1940s.

This report attempts to detect likely matches using the heuristics above (presence of "collected" / "volume" with "Batman"). If the exact row is present it will be flagged and included in the flagged items table.

If the item exists and is misidentified:
1. Manually correct the `Series` / `Title` / `Issue Number` entry in the CSV so it clearly reads the correct series/volume (e.g., `Batman: The Collected Adventures - Vol. 1`).
2. If the CSV is updated, re-run this Quarto file to recompute totals.
3. For high-value disagreements (e.g., the system valued a collected volume as a 1940s single issue), remove or correct the row before using the collection total — the conservative total excluding flagged items is intended to give a safer baseline until those reviews are completed.

## Final conservative estimate (conclusion)


In [ ]:
#| label: final-summary
print()
print("FINAL CONSERVATIVE ESTIMATE SUMMARY")
print("-" * 40)
print(f"Number of items processed: {total_items}")
print(f"Conservative total (all low estimates): ${conservative_total_all:,.2f}")
print(f"Conservative total (excluding flagged items): ${conservative_total_unflagged:,.2f}")
print()
print("Recommendation:")
print(textwrap.fill(\"\"\"\nUse the 'excluding flagged items' total as the conservative estimate until flagged items have been manually reviewed and corrected. Flagged items include potential misidentifications (e.g., collected volumes labeled as single issues) and low-confidence identifications.\n\"\"\", width=80))

## Notes

- This summary intentionally uses conservative logic (sum of low estimates) to avoid overstating value.
- Automated classification is imperfect; high-value anomalies (e.g., a collected edition misidentified as a rare first issue) must be corrected manually.
- If you want, I can:
  - Add a small interactive notebook to inspect flagged images and edit confirmations inline.
  - Add automatic re-grounding steps (re-query sources) for flagged items to try to improve identification confidence.